In [ ]:
import shap
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import statsmodels.api as sm
import pygraphviz as pgv
from sklearn.impute import SimpleImputer
from IPython.display import Image, display

# Redefine the fixed causal graph to ensure it's the correct structure
fixed_causal_graph_robust = nx.DiGraph([G])


# --- Data Cleaning for SHAP Calculation --- (Copied for consistency)
df_cleaned_for_shap_robust = df.copy()
df_cleaned_for_shap_robust = df_cleaned_for_shap_robust.replace([np.inf, -np.inf], np.nan)
imputer = SimpleImputer(strategy='mean')
numerical_cols_for_imputation_robust = df_cleaned_for_shap_robust.select_dtypes(include=np.number).columns.tolist()
df_cleaned_for_shap_robust[numerical_cols_for_imputation_robust] = imputer.fit_transform(df_cleaned_for_shap_robust[numerical_cols_for_imputation_robust])

print(f"Number of nodes in fixed_causal_graph_robust: {len(fixed_causal_graph_robust.nodes())}")

# Step 2: Compute SHAP values for each edge using Robust Linear Model (RLM)
robust_fixed_graph_shap_strengths = {}
print("--- Starting SHAP calculation loop for Fixed Graph with RLM ---")

for child_node in fixed_causal_graph_robust.nodes():
    if child_node not in df_cleaned_for_shap_robust.columns:
        continue

    parents = list(fixed_causal_graph_robust.predecessors(child_node))
    if not parents:
        continue

    valid_parents = [p for p in parents if p in df_cleaned_for_shap_robust.columns and pd.api.types.is_numeric_dtype(df_cleaned_for_shap_robust[p])]
    if not valid_parents:
        continue

    X_parents = df_cleaned_for_shap_robust[valid_parents]
    y_child = df_cleaned_for_shap_robust[child_node]

    if y_child.nunique() == 1:
        continue

    constant_cols_in_X = [col for col in X_parents.columns if X_parents[col].nunique() == 1]
    if constant_cols_in_X:
        X_parents = X_parents.drop(columns=constant_cols_in_X)
        valid_parents = [p for p in valid_parents if p not in constant_cols_in_X]
        if not valid_parents:
            continue

    X_parents_const = sm.add_constant(X_parents)

    try:
        # Use Robust Linear Model (RLM) for robust regression
        rlm_model = sm.RLM(y_child, X_parents_const, M=sm.robust.norms.HuberT()).fit()
        print(f"  RLM model fit successful for '{child_node}'.")
    except Exception as e:
        print(f"  Debug: Could not fit RLM model for child '{child_node}' with parents {valid_parents}. Error: {e}. Skipping SHAP calculation.")
        continue

    if len(X_parents_const) > 100:
        background = shap.sample(X_parents_const, 100)
    else:
        background = X_parents_const

    explainer = shap.KernelExplainer(rlm_model.predict, background)
    print(f"  KernelExplainer created for '{child_node}' using RLM. Calculating SHAP values...")

    shap_values_raw_result = explainer.shap_values(X_parents_const, nsamples=100)
    if isinstance(shap_values_raw_result, list):
        shap_values_raw = shap_values_raw_result[0]
    else:
        shap_values_raw = shap_values_raw_result

    print(f"  SHAP values calculated for '{child_node}'.")

    for i, p in enumerate(valid_parents):
        robust_fixed_graph_shap_strengths[(p, child_node)] = np.mean(np.abs(shap_values_raw[:, i + 1]))
    print(f"  Added SHAP strengths for '{child_node}' to dictionary.")

print("--- End SHAP calculation loop for Fixed Graph with RLM ---")
print("Calculated RLM SHAP-based edge strengths for Fixed Graph:")
print(robust_fixed_graph_shap_strengths)

# Step 3: Normalize SHAP strengths for visualization (penwidth) - 0.5 to 5 scale
normalized_robust_fixed_graph_shap_values = {}
if robust_fixed_graph_shap_strengths:
    shap_values_array_robust = np.array(list(robust_fixed_graph_shap_strengths.values())).reshape(-1, 1)
    non_zero_shap_values_robust = shap_values_array_robust[shap_values_array_robust != 0]

    if len(non_zero_shap_values_robust) == 0:
        normalized_robust_fixed_graph_shap_values = {edge: 0.5 for edge in robust_fixed_graph_shap_strengths}
    elif np.max(non_zero_shap_values_robust) - np.min(non_zero_shap_values_robust) < 1e-9:
        normalized_robust_fixed_graph_shap_values = {edge: 0.5 for edge in robust_fixed_graph_shap_strengths if robust_fixed_graph_shap_strengths[edge] != 0}
        for edge in robust_fixed_graph_shap_strengths:
            if robust_fixed_graph_shap_strengths[edge] == 0:
                normalized_robust_fixed_graph_shap_values[edge] = 0.0
    else:
        scaler_robust = MinMaxScaler(feature_range=(0.5, 5))
        temp_shap_values_for_scaling_robust = np.array([v for v in robust_fixed_graph_shap_strengths.values() if v != 0]).reshape(-1, 1)
        if len(temp_shap_values_for_scaling_robust) > 0:
            scaled_values_robust = scaler_robust.fit_transform(temp_shap_values_for_scaling_robust).flatten()
            scaled_idx_robust = 0
            for edge in robust_fixed_graph_shap_strengths:
                if robust_fixed_graph_shap_strengths[edge] != 0:
                    normalized_robust_fixed_graph_shap_values[edge] = scaled_values_robust[scaled_idx_robust]
                    scaled_idx_robust += 1
                else:
                    normalized_robust_fixed_graph_shap_values[edge] = 0.0
        else:
            normalized_robust_fixed_graph_shap_values = {edge: 0.0 for edge in robust_fixed_graph_shap_strengths}
else:
    print("No robust fixed graph SHAP strengths to normalize.")

# Step 4: Visualize the causal graph with Robust SHAP-scaled edge strengths
if fixed_causal_graph_robust.number_of_edges() > 0:
    A_robust = nx.nx_agraph.to_agraph(fixed_causal_graph_robust)

    A_robust.graph_attr['rankdir'] = 'LR'
    A_robust.node_attr['shape'] = 'box'
    A_robust.node_attr['fontsize'] = '10'
    A_robust.edge_attr['fontsize'] = '8'

    for u, v in fixed_causal_graph_robust.edges():
        normalized_strength = normalized_robust_fixed_graph_shap_values.get((u, v), 0.0)
        original_shap = robust_fixed_graph_shap_strengths.get((u, v), 0.0)

        edge = A_robust.get_edge(str(u), str(v))
        if edge:
            if normalized_strength > 0:
                edge.attr['penwidth'] = str(normalized_strength)
                edge.attr['label'] = f'{original_shap:.2f}'
            else:
                edge.attr['penwidth'] = '0.1'
                edge.attr['color'] = 'lightgray'
                edge.attr['label'] = ''
        else:
            print(f"Warning: Edge ({u}, {v}) not found in AGraph when adding attributes.")

    output_filename_robust_shap = 'Causal_Graph_Robust_SHAP_Fixed.png'
    A_robust.draw(output_filename_robust_shap, prog='dot')

    print(f"Causal graph with Robust SHAP-scaled edge strengths saved as '{output_filename_robust_shap}'")

    display(Image(output_filename_robust_shap))
else:
    print("Fixed causal graph has no edges to visualize.")
